# Surgical Concept Erasure: Interactive Demo

This notebook runs the full pipeline from the `concept-surgery` repo on a free Colab GPU (or CPU): map a concept's direction inside GPT-2, fit it, select the best layer/estimator, ablate it, and measure before/after.

In [ ]:
!pip -q install torch transformers accelerate scikit-learn scipy
!git clone https://github.com/YOUR_USERNAME/concept-surgery.git
%cd concept-surgery

## 1. Baseline behavior

In [ ]:
import sys; sys.path.insert(0, ".")
from concept_surgery.capture import load_model_and_tokenizer
from concept_surgery.data import load_prompts
from concept_surgery.evaluate import generate

model, tokenizer, device = load_model_and_tokenizer("gpt2")
concept = load_prompts("mickey_mouse")
neutral = load_prompts("neutral")

for p in ["Tell me a story about Mickey Mouse.", "Explain how rainbows form."]:
    print(f"[prompt] {p}\n[before] {generate(model, tokenizer, p, device, 60)}\n")

## 2. Run the pipeline (map → fit → select → ablate → prove)

In [ ]:
from concept_surgery.config import ErasureConfig
from concept_surgery.pipeline import run_pipeline

cfg = ErasureConfig(model_name="gpt2", concept_dataset="mickey_mouse", output_dir="runs/colab")
results = run_pipeline(cfg)

## 3. Scorecard

In [ ]:
import json
b, a = results["baseline"], results["ablated"]
print(f"probe accuracy   : {b['probe_acc']:.3f} -> {a['probe_acc']:.3f}  (chance = 0.5)")
print(f"concept NLL      : {b['concept_nll']:.3f} -> {a['concept_nll']:.3f}")
print(f"neutral NLL      : {b['neutral_nll']:.3f} -> {a['neutral_nll']:.3f}  (collateral damage)")
print(f"locality KL      : {results['locality_kl']:.4f}  (0 = perfectly surgical)")
print(f"chosen           : layer {results['chosen']['layer']}, method {results['chosen']['method']}")

## 4. After the surgery

In [ ]:
for row in results["generations"][:6]:
    print(f"[prompt] {row['prompt']}\n[before] {row['baseline'][:200]}\n[after ] {row['ablated'][:200]}\n")

## 5. Go further

- Swap `--concept harry_potter`, or pass your own JSONL.
- Compare estimators: `--methods mean_diff leace inlp`.
- Try multi-layer ablation: `--layers 8 9 10` on a bigger model.
- Read `paper/paper.md` for the math and the honest limitations.